In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

base_dir = Path(r"F:\رواد\Smart-Factory-Monitoring-Platform-main\Smart-Factory-Monitoring-Platform-main\Data")
output_path = base_dir / "merged_output.csv"

files = [
    "errors.csv",
    "failures.csv",
    "failures_features.csv",
    "machines.csv",
    "maint.csv",
    "telemetry.csv"
]

row_counts = {}
for file_name in files:
    with open(base_dir / file_name, "r", encoding="utf-8") as f:
        row_counts[file_name] = sum(1 for _ in f) - 1

max_file = max(row_counts, key=row_counts.get)
chunk_size = 50000

cached = {}
for file_name in files:
    if file_name != max_file:
        cached[file_name] = pd.read_csv(base_dir / file_name, dtype=str, keep_default_na=False, low_memory=False)

if output_path.exists():
    output_path.unlink()

base_iter = pd.read_csv(base_dir / max_file, dtype=str, keep_default_na=False, low_memory=False, chunksize=chunk_size)

start = 0
first_chunk = True

for base_chunk in base_iter:
    end = start + len(base_chunk)
    parts = []

    for file_name in files:
        prefix = f"{Path(file_name).stem}_"
        if file_name == max_file:
            part = base_chunk.reset_index(drop=True).add_prefix(prefix)
        else:
            df = cached[file_name]
            idx = np.arange(start, end) % len(df)
            part = df.iloc[idx].reset_index(drop=True).add_prefix(prefix)
        parts.append(part)

    merged_chunk = pd.concat(parts, axis=1)
    merged_chunk.to_csv(output_path, mode="a", index=False, header=first_chunk)
    first_chunk = False
    start = end

print(f"Saved to: {output_path}")